In [2]:
import os
import torch
import torch.nn
from torch.optim import Adam
from torchvision.transforms import transforms
import pandas as pd
from torch.utils.data import DataLoader,Dataset
from sklearn.preprocessing import LabelEncoder
from PIL import Image

In [3]:
from torchvision import models

In [49]:
resnet50=models.resnet50(weights='DEFAULT')
resnet50.fc

Linear(in_features=2048, out_features=1000, bias=True)

In [30]:
image_path=[]
labels=[]

for i in os.listdir(f"/kaggle/input/datasets/mohiuddinmahady/counterfeit-currency-bangladesh/Bangladeshi Counterfeit Currency Image Dataset/Denomination-wise Distribution"):
    for label in os.listdir(f"/kaggle/input/datasets/mohiuddinmahady/counterfeit-currency-bangladesh/Bangladeshi Counterfeit Currency Image Dataset/Denomination-wise Distribution/{i}"):
        labels.append(i)
        image_path.append(f'/kaggle/input/datasets/mohiuddinmahady/counterfeit-currency-bangladesh/Bangladeshi Counterfeit Currency Image Dataset/Denomination-wise Distribution/{i}/{label}')

data_df=pd.DataFrame(zip(image_path,labels),columns=['image_path','labels'])
data_df.head()

,image_path,labels
0,/kaggle/input/datasets/mohiuddinmahady/counter...,1000 BDT Genuine
1,/kaggle/input/datasets/mohiuddinmahady/counter...,1000 BDT Genuine
2,/kaggle/input/datasets/mohiuddinmahady/counter...,1000 BDT Genuine
3,/kaggle/input/datasets/mohiuddinmahady/counter...,1000 BDT Genuine
4,/kaggle/input/datasets/mohiuddinmahady/counter...,1000 BDT Genuine


In [6]:
data_df.shape

(1286, 2)

In [31]:
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [32]:
train_df=data_df.sample(frac=0.7,random_state=42)
val_df=data_df.drop(train_df.index)

In [33]:
val_df.shape

(386, 2)

In [44]:
label_encoder=LabelEncoder()
label_encoder.fit(data_df['labels'])

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.2),
    transforms.ToTensor(),
    transforms.RandomResizedCrop(128,scale=(0.8,1.0)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


In [45]:
class MyDataset(Dataset):
    def __init__(self,dataframe,transform):
        self.dataframe=dataframe
        self.transform=transform
        self.labels=torch.tensor(label_encoder.transform(self.dataframe['labels'])).to(device)


    def __len__(self):
        return self.dataframe.shape[0]


    def __getitem__(self,idx):
        img=self.dataframe.iloc[idx,0]
        label=self.labels[idx]
        img=Image.open(img).convert("RGB")

        if self.transform:
            img=(self.transform(img)).to(device)
        return img,label

In [46]:
train_dataset=MyDataset(train_df,transform=train_transform)
val_dataset=MyDataset(val_df,transform=val_transform)

In [47]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False)

In [50]:
num_features=len(data_df['labels'].unique())
# for params in resnet50.parameters():
    # params.requires_grad=False
resnet50.fc=torch.nn.Sequential(
    torch.nn.Linear(resnet50.fc.in_features,out_features= 256),
    torch.nn.ReLU(),
    torch.nn.Dropout(0.5),
    torch.nn.Linear(256,num_features)
)

resnet50.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [51]:
criterion=torch.nn.CrossEntropyLoss()
optimizer=Adam(resnet50.parameters(),lr=0.00001)
epochs=50

In [52]:
total_train_loss_plot=[]
total_val_loss_plot=[]
total_train_acc_plot=[]
total_val_acc_plot=[]
best_val_loss=float('inf')
patience=5
counter=0
print("Training Started")

for epoch in range(epochs):
    resnet50.train()
    total_train_loss=0.0
    total_train_acc=0.0
    total_val_loss=0.0
    total_val_acc=0.0

    for inputs,labels in train_loader:
        optimizer.zero_grad()
        pred=resnet50(inputs)
        acc=(torch.argmax(pred,axis=1)==labels).sum().item()
        loss=criterion(pred,labels)
        loss.backward()
        optimizer.step()
        total_train_acc+=acc
        total_train_loss+=loss.item()

    resnet50.eval()
    with torch.inference_mode():
        for inputs,labels in val_loader:
            pred=resnet50(inputs)
            acc=(torch.argmax(pred,axis=1)==labels).sum().item()
            loss=criterion(pred,labels)
    
            total_val_loss+=loss.item()
            total_val_acc+=acc

    current_val_loss=total_val_loss/len(val_loader)
    current_val_acc=(100*total_val_acc/len(val_dataset))
    total_train_loss_plot.append(total_train_loss/len(train_loader))
    total_train_acc_plot.append(100*total_train_acc/len(train_dataset))
    total_val_acc_plot.append(current_val_acc)
    total_val_loss_plot.append(current_val_loss)
    
    print("="*25)
    print(f"For {epoch}/{epochs} Train Loss:{total_train_loss_plot[-1]:.4f} Train Accuracy:{total_train_acc_plot[-1]:.4f} ")
    print(f"For {epoch}/{epochs} Val Loss:{total_val_loss_plot[-1]:.4f} Val Accuracy:{total_val_acc_plot[-1]:.4f} ")

    if best_val_loss>current_val_loss:
        best_val_loss=current_val_loss
        counter=0
        torch.save({
            'epoch':epoch,
            'val_loss':current_val_loss,
            'val_acc':current_val_acc,
            'model_state_dict':resnet50.state_dict()
        },'best_model.pth')
        print("Best Model Saved")
        
    else:
        counter+=1
        print(f"Value of counter is:{counter}/{patience}")
        if counter>=patience:
            print("Early Stopping")
            break
        

Training Started
For 0/50 Train Loss:1.3595 Train Accuracy:33.0000 
For 0/50 Val Loss:1.3531 Val Accuracy:50.2591 
Best Model Saved
For 1/50 Train Loss:1.2287 Train Accuracy:65.6667 
For 1/50 Val Loss:1.2267 Val Accuracy:68.3938 
Best Model Saved
For 2/50 Train Loss:1.0565 Train Accuracy:70.5556 
For 2/50 Val Loss:1.0884 Val Accuracy:68.1347 
Best Model Saved
For 3/50 Train Loss:0.8591 Train Accuracy:73.6667 
For 3/50 Val Loss:0.9889 Val Accuracy:68.3938 
Best Model Saved
For 4/50 Train Loss:0.7061 Train Accuracy:76.1111 
For 4/50 Val Loss:0.8931 Val Accuracy:69.4301 
Best Model Saved
For 5/50 Train Loss:0.5811 Train Accuracy:82.2222 
For 5/50 Val Loss:0.8043 Val Accuracy:77.2021 
Best Model Saved
For 6/50 Train Loss:0.4797 Train Accuracy:86.4444 
For 6/50 Val Loss:0.7385 Val Accuracy:86.2694 
Best Model Saved
For 7/50 Train Loss:0.3769 Train Accuracy:91.2222 
For 7/50 Val Loss:0.6815 Val Accuracy:93.2642 
Best Model Saved
For 8/50 Train Loss:0.3147 Train Accuracy:93.2222 
For 8/50 Val

In [55]:
import joblib

In [56]:
joblib.dump(label_encoder,"label_encoder.pkl")

['label_encoder.pkl']

In [57]:
from IPython.display import FileLink
FileLink('label_encoder.pkl')

/kaggle/working/label_encoder.pkl